In [1]:
#imports and parameters used
from Bio.SeqIO.FastaIO import SimpleFastaParser 
import pandas as pd
import json
from collections import deque
import os

# Step 1 - MAX algorithms
MIN_ALGORITHM_MATCH = 9

# Step 2 - Calculate BLAST-P
min_blast_identity = 21 # 21 in paper


SAVE_VARIANT_DF_TO_CSV = True



# STEP 1: Max-algorithm sorting

In [2]:
AGR_ortholog_data_path = r"downloaded_data\ORTHOLOGY_AGR.json"

with open(AGR_ortholog_data_path, "r", encoding="utf-8") as f:
    AGR_ortholog_data = json.load(f)

ortholog_df = pd.DataFrame(AGR_ortholog_data["data"])


human_worm_orthologs = ortholog_df[
    (ortholog_df["Gene1SpeciesName"] == "Homo sapiens") &
    (ortholog_df["Gene2SpeciesName"] == "Caenorhabditis elegans")
] 

# renaming gene1 to human and gene2 to worm
human_worm_orthologs.columns = human_worm_orthologs.columns.str.replace("Gene1", "Human_", regex=False)
human_worm_orthologs.columns = human_worm_orthologs.columns.str.replace("Gene2", "Worm_", regex=False)

# removing WB:WBxxxx prefix
human_worm_orthologs["Worm_ID"] = human_worm_orthologs["Worm_ID"].str.replace({"WB:":""})

#AlgorithmsMatch was str
human_worm_orthologs["AlgorithmsMatch"] = human_worm_orthologs["AlgorithmsMatch"].astype(int)
human_worm_orthologs["OutOfAlgorithms"] = human_worm_orthologs["OutOfAlgorithms"].astype(int)


# only maximal algorithms matches
if MIN_ALGORITHM_MATCH < 9:
    orthologs_df = human_worm_orthologs[
    (human_worm_orthologs["AlgorithmsMatch"] >= MIN_ALGORITHM_MATCH)
]
    print(f"Using min of MIN_ALGORITHM_MATCH of: {MIN_ALGORITHM_MATCH}")
else:
    orthologs_df = human_worm_orthologs[
    (human_worm_orthologs["AlgorithmsMatch"] == human_worm_orthologs["OutOfAlgorithms"])
    ]
    print("Using max of OutOfAlgorithms")

display(orthologs_df)
count_before_review_filtering = orthologs_df.shape[0]

# human_ids = orthologs_df["Human_ID"].dropna().unique().tolist()

# with open("all_HGNC_ids_from_AGR.txt", "w") as text_file:
#     for human_id in human_ids:
#         text_file.write(f"{human_id}\n")

Using max of OutOfAlgorithms


,Human_ID,Human_Symbol,Human_SpeciesTaxonID,Human_SpeciesName,Worm_ID,Worm_Symbol,Worm_SpeciesTaxonID,Worm_SpeciesName,Algorithms,AlgorithmsMatch,OutOfAlgorithms,IsBestScore,IsBestRevScore
749240,HGNC:117,ACO1,NCBITaxon:9606,Homo sapiens,WBGene00000040,aco-1,NCBITaxon:6239,Caenorhabditis elegans,"[OrthoInspector, PANTHER, Ensembl Compara, Ort...",9,9,Yes,Yes
749248,HGNC:118,ACO2,NCBITaxon:9606,Homo sapiens,WBGene00000041,aco-2,NCBITaxon:6239,Caenorhabditis elegans,"[OrthoInspector, InParanoid, OMA, Ensembl Comp...",9,9,Yes,Yes
749482,HGNC:240,ADCY9,NCBITaxon:9606,Homo sapiens,WBGene00000068,acy-1,NCBITaxon:6239,Caenorhabditis elegans,"[PANTHER, PhylomeDB, OMA, InParanoid, Hieranoi...",9,9,Yes,Yes
749511,HGNC:237,ADCY6,NCBITaxon:9606,Homo sapiens,WBGene00000071,acy-4,NCBITaxon:6239,Caenorhabditis elegans,"[Hieranoid, SonicParanoid, OrthoInspector, PAN...",9,9,Yes,Yes
749624,HGNC:327,AGPS,NCBITaxon:9606,Homo sapiens,WBGene00000081,ads-1,NCBITaxon:6239,Caenorhabditis elegans,"[InParanoid, OrthoFinder, Hieranoid, OrthoInsp...",9,9,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
847529,HGNC:4061,SLC37A4,NCBITaxon:9606,Homo sapiens,WBGene00302978,C42C1.19,NCBITaxon:6239,Caenorhabditis elegans,"[Ensembl Compara, OMA, PhylomeDB, OrthoInspect...",9,9,Yes,Yes
847552,HGNC:28419,SMIM7,NCBITaxon:9606,Homo sapiens,WBGene00302984,Y51H7C.41,NCBITaxon:6239,Caenorhabditis elegans,"[Hieranoid, Ensembl Compara, PhylomeDB, InPara...",9,9,Yes,Yes
847591,HGNC:6937,MCCC2,NCBITaxon:9606,Homo sapiens,WBGene00303021,mccc-2,NCBITaxon:6239,Caenorhabditis elegans,"[SonicParanoid, InParanoid, PANTHER, Ensembl C...",9,9,Yes,Yes
847597,HGNC:30187,TMEM167B,NCBITaxon:9606,Homo sapiens,WBGene00303023,K07F5.22,NCBITaxon:6239,Caenorhabditis elegans,"[Hieranoid, SonicParanoid, InParanoid, OrthoFi...",9,9,Yes,Yes


# STEP 2: Finding FASTA sequecne matches for human and C.elegans genes via:
https://parasite.wormbase.org/Caenorhabditis_elegans_prjna13758/Info/Index/

In [3]:
from functions.fasta_related import fasta_data_to_fasta_df
from functions.fasta_related import keep_longest_seq_per_human_uniprot, keep_reviewed_longest_worm_per_gene

worm_fasta_data_path = r"downloaded_data\CelegansFASTAsequences.fa"

human_fasta_data_path = r"downloaded_data\Human_FASTAs_and_uniprot_ids.tsv"


worm_FASTA_df = fasta_data_to_fasta_df(worm_fasta_data_path, species="c_elegans")
human_FASTA_df = pd.read_csv(
    human_fasta_data_path,
    sep="\t", encoding="utf-8"
)
human_FASTA_df = human_FASTA_df.rename(columns={"Entry":"uniprot_id", "Sequence": "FASTA Sequence", "From":"HGNC"})

display(human_FASTA_df.head())

worm_FASTA_df = worm_FASTA_df.dropna(subset=["FASTA Sequence"])
human_FASTA_df = human_FASTA_df.dropna(subset=["FASTA Sequence"])

display(worm_FASTA_df.head())

worm_FASTA_df = worm_FASTA_df[["gene", "FASTA Sequence", "status"]].drop_duplicates()
human_fasta_df = human_FASTA_df[["HGNC","uniprot_id", "FASTA Sequence"]].drop_duplicates()

display(human_fasta_df.head())


# There are multiple FASTA sequence for some genes, based  on the Transcript.
# We filter based on two criteria, has status "Confirmed" and is the longest sequence for a given gene.
# human has no status to look for, so just longest gene for human
human_fasta_df = keep_longest_seq_per_human_uniprot(human_fasta_df)
worm_FASTA_df = keep_reviewed_longest_worm_per_gene(worm_FASTA_df)


# merging fasta and orthologs_df

orthologs_df = orthologs_df.merge(
    human_fasta_df,
    left_on="Human_ID", # Human_ID is the HGNC for ensembl database
    right_on="HGNC",
    how="left"
).rename(columns={"FASTA Sequence": "Human_FASTA Sequence",})

orthologs_df = orthologs_df.merge(
    worm_FASTA_df,
    left_on="Worm_ID",
    right_on="gene", # gene is fx: WBGene00007063
    how="left"
).rename(columns={"FASTA Sequence": "Worm_FASTA Sequence",})

orthologs_df = orthologs_df.dropna(subset=["Worm_FASTA Sequence"])



orthologs_df.head(10)
orthologs_df.shape[0]

,HGNC,uniprot_id,Entry Name,Protein names,Gene Names,Organism,Length,FASTA Sequence
0,HGNC:8980,O00459,P85B_HUMAN,Phosphatidylinositol 3-kinase regulatory subun...,PIK3R2,Homo sapiens (Human),728,MAGPEGFQYRALYPFRRERPEDLELLPGDVLVVSRAALQALGVAEG...
1,HGNC:8981,Q92569,P55G_HUMAN,Phosphatidylinositol 3-kinase regulatory subun...,PIK3R3,Homo sapiens (Human),461,MYNTVWSMDRDDADWREVMMPYSTELIFYIEMDPPALPPKPPKPMT...
2,HGNC:8979,P27986,P85A_HUMAN,Phosphatidylinositol 3-kinase regulatory subun...,PIK3R1 GRB1,Homo sapiens (Human),724,MSAEGYQYRALYDYKKEREEDIDLHLGDILTVNKGSLVALGFSDGQ...
3,HGNC:11066,Q9UHI5,LAT2_HUMAN,Large neutral amino acids transporter small su...,SLC7A8 LAT2,Homo sapiens (Human),535,MEEGARHRNNTEKKHPGGGESDASPEAGSGGGGVALKKEIGLVSAC...
4,HGNC:11059,Q9UPY5,XCT_HUMAN,Cystine/glutamate transporter (Amino acid tran...,SLC7A11,Homo sapiens (Human),501,MVRKPVVSTISKGGYLQGNVNGRLPSLGNKEPPGQEKVQLKRKVTL...


,FASTA ID,Header,FASTA Sequence,wormpep,gene,status,uniprot,insdc,product,locus
0,2L52.1a,2L52.1a wormpep=CE32090 gene=WBGene00007063 st...,MSMVRNVSNQSEKLEILSCKWVGCLKSTEVFKTVEKLLDHVTADHI...,CE32090,WBGene00007063,Confirmed,A4F336,CCD61130.1,C2H2-type domain-containing protein,NaN
1,2L52.1b,2L52.1b wormpep=CE50569 gene=WBGene00007063 st...,MSDNEEVYVNFRGMNCISTGKSASMVPSKRRNWPKRVKKRLSTQRN...,CE50569,WBGene00007063,Confirmed,A0A0K3AWR5,CTQ86426.1,C2H2-type domain-containing protein,NaN
2,2RSSE.1a,2RSSE.1a wormpep=CE32785 gene=WBGene00007064 l...,MTVASYSMVLCGSSDDHRYRGRIEKVKFGVPINEAFAHDIPATLLM...,CE32785,WBGene00007064,Confirmed,A4F337,CCD61138.1,Rho-GAP domain-containing protein,rga-9
3,2RSSE.1b,2RSSE.1b wormpep=CE48524 gene=WBGene00007064 l...,MTVASYSMVLCGSSDDHRYRGRIEKVKFGVPINEAFAHDIPATLLM...,CE48524,WBGene00007064,Confirmed,U4PAK0,CDH92922.1,Rho-GAP domain-containing protein,rga-9
4,2RSSE.1c,2RSSE.1c wormpep=CE52653 gene=WBGene00007064 l...,MTVASYSMVLCGSSDDHRYRGRIEKVKFGVPINEAFAHDIPATLLM...,CE52653,WBGene00007064,Confirmed,A0A2X0T1Z3,SPS41576.1,Rho-GAP domain-containing protein,rga-9


,HGNC,uniprot_id,FASTA Sequence
0,HGNC:8980,O00459,MAGPEGFQYRALYPFRRERPEDLELLPGDVLVVSRAALQALGVAEG...
1,HGNC:8981,Q92569,MYNTVWSMDRDDADWREVMMPYSTELIFYIEMDPPALPPKPPKPMT...
2,HGNC:8979,P27986,MSAEGYQYRALYDYKKEREEDIDLHLGDILTVNKGSLVALGFSDGQ...
3,HGNC:11066,Q9UHI5,MEEGARHRNNTEKKHPGGGESDASPEAGSGGGGVALKKEIGLVSAC...
4,HGNC:11059,Q9UPY5,MVRKPVVSTISKGGYLQGNVNGRLPSLGNKEPPGQEKVQLKRKVTL...


2992

# STEP 3: BLAST-P calculation

In [4]:
from functions.blast_p_calculator import blast_p_calculation # requires local blast

display(orthologs_df)

orthologs_df = blast_p_calculation(
    df=orthologs_df,
    num_threads = 8
)
orthologs_df["BLAST_pident"]

,Human_ID,Human_Symbol,Human_SpeciesTaxonID,Human_SpeciesName,Worm_ID,Worm_Symbol,Worm_SpeciesTaxonID,Worm_SpeciesName,Algorithms,AlgorithmsMatch,...,IsBestScore,IsBestRevScore,HGNC,uniprot_id,Human_FASTA Sequence,Human_protein_length,gene,Worm_FASTA Sequence,status,Worm_protein_length
0,HGNC:117,ACO1,NCBITaxon:9606,Homo sapiens,WBGene00000040,aco-1,NCBITaxon:6239,Caenorhabditis elegans,"[OrthoInspector, PANTHER, Ensembl Compara, Ort...",9,...,Yes,Yes,HGNC:117,P21399,MSNPFAHLAEPLDPVQPGKKFFNLNKLEDSRYGRLPFSIRVLLEAA...,889.0,WBGene00000040,MAFNNLIRNLAIGDNVYKYFDLNGLNDARYNELPISIKYLLEAAVR...,Confirmed,887.0
1,HGNC:118,ACO2,NCBITaxon:9606,Homo sapiens,WBGene00000041,aco-2,NCBITaxon:6239,Caenorhabditis elegans,"[OrthoInspector, InParanoid, OMA, Ensembl Comp...",9,...,Yes,Yes,HGNC:118,Q99798,MAPYSLLVTRLQKALGVRQYHVASVLCQRAKVAMSHFEPNEYIHYD...,780.0,WBGene00000041,MNSLLRLSHLAGPAHYRALHSSSSIWSKVAISKFEPKSYLPYEKLS...,Confirmed,777.0
2,HGNC:240,ADCY9,NCBITaxon:9606,Homo sapiens,WBGene00000068,acy-1,NCBITaxon:6239,Caenorhabditis elegans,"[PANTHER, PhylomeDB, OMA, InParanoid, Hieranoi...",9,...,Yes,Yes,HGNC:240,O60503,MASPPHQQLLHHHSTEVSCDSSGDSNSVRVKINPKQLSSNSHPKHC...,1353.0,WBGene00000068,MDDDVGERTPALGGSCGPSVRAHSSSPRRVPLFERASARWWNPQFR...,Confirmed,1253.0
4,HGNC:327,AGPS,NCBITaxon:9606,Homo sapiens,WBGene00000081,ads-1,NCBITaxon:6239,Caenorhabditis elegans,"[InParanoid, OrthoFinder, Hieranoid, OrthoInsp...",9,...,Yes,Yes,HGNC:327,O00116,MAEAAAAAGGTGLGAGASYGSAADRDRDPDPDRAGRRLRVLSGHLL...,658.0,WBGene00000081,MSASYQTIEHDVPQSYRDKILKWNGWGYSDSQFAINKDGHVTFTGD...,Confirmed,597.0
5,HGNC:6766,MADD,NCBITaxon:9606,Homo sapiens,WBGene00000086,aex-3,NCBITaxon:6239,Caenorhabditis elegans,"[PhylomeDB, OrthoFinder, Ensembl Compara, OMA,...",9,...,Yes,Yes,HGNC:6766,Q8WXG6,MVQKKKFCPRLLDYLVIVGARHPSSDSVAQTPELLRRYPLEDHTEF...,1647.0,WBGene00000086,MNDKEKEICPRLIDFLVVVGKRNRTRGASQSSPDATTDTTVTYPEI...,Confirmed,1409.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3107,HGNC:30551,TXNL4A,NCBITaxon:9606,Homo sapiens,WBGene00235102,dib-1,NCBITaxon:6239,Caenorhabditis elegans,"[PANTHER, SonicParanoid, InParanoid, Hieranoid...",9,...,Yes,Yes,HGNC:30551,P83876,MSYMLPHLHNGWQVDQAILSEEDRVVVIRFGHDWDPTCMKMDEVLY...,142.0,WBGene00235102,MSYMLPHLENGWQVDQAILAEEDRVVVVRFGHDWDPTCMQMDETLF...,Confirmed,142.0
3108,HGNC:23301,PBLD,NCBITaxon:9606,Homo sapiens,WBGene00235310,R02D1.2,NCBITaxon:6239,Caenorhabditis elegans,"[OrthoFinder, PANTHER, SonicParanoid, Hieranoi...",9,...,Yes,Yes,HGNC:23301,P30039,MKLPIFIADAFTARAFRGNPAAVCLLENELDEDMHQKIAREMNLSE...,288.0,WBGene00235310,MPKNYPSFIVDAFTKTSFGGNPAAVCLIPKALSDREYLKISSEFNL...,Confirmed,317.0
3110,HGNC:4061,SLC37A4,NCBITaxon:9606,Homo sapiens,WBGene00302978,C42C1.19,NCBITaxon:6239,Caenorhabditis elegans,"[Ensembl Compara, OMA, PhylomeDB, OrthoInspect...",9,...,Yes,Yes,HGNC:4061,O43826,MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDK...,429.0,WBGene00302978,MATSPTNYFKFGAVFCIYVAYVLFRRLPIAFLSQIRTEIQISNDDI...,Confirmed,403.0
3111,HGNC:28419,SMIM7,NCBITaxon:9606,Homo sapiens,WBGene00302984,Y51H7C.41,NCBITaxon:6239,Caenorhabditis elegans,"[Hieranoid, Ensembl Compara, PhylomeDB, InPara...",9,...,Yes,Yes,HGNC:28419,Q9BQ49,MIGDILLFGTLLMNAGAVLNFKLKKKDTQGFGEESREPSTGDNIRE...,75.0,WBGene00302984,MLSELIIAGTLVINAFAVLNFNLKKSPENAFGVVETHQTDSLGDKV...,Confirmed,77.0


0       63.982
1       74.507
2       45.851
3       53.927
4       42.290
         ...  
2987    86.620
2988    37.097
2989    26.559
2990    50.649
2991    66.667
Name: BLAST_pident, Length: 2992, dtype: float64

# STEP 4: Add variant information (human gene)

In [ ]:
from functions.variant_querying import add_variants

os.makedirs("created_data", exist_ok=True)

if SAVE_VARIANT_DF_TO_CSV:

    save_path = f"created_data/gene_with_variant_data_using_{MIN_ALGORITHM_MATCH}_algorithms.csv"

    if os.path.exists(save_path):
        print(f"File already exists, skipping pipeline: {save_path}")
        variant_df = pd.read_csv(save_path)
        variant_df = variant_df.rename(columns={"consequence": "VEP ANNO"})
        variant_df = variant_df.drop_duplicates()

    else:
        add_variants(orthologs_df, variant_database=save_path)
        backup_path = f"created_data/gene_with_variant_data_using_{MIN_ALGORITHM_MATCH}_algorithms_backup.csv"

        print(f"Saving variant backup to csv file: {backup_path}")
        variant_df = pd.read_csv(save_path)
        variant_df = variant_df.rename(columns={"consequence": "VEP ANNO"})

        variant_df.to_csv(backup_path, index=False)
else:
    print("running without saving")
    save_path = "empty.csv"
    add_variants(orthologs_df, variant_database="empty.csv")
    variant_df = pd.read_csv(save_path)
    variant_df = variant_df.rename(columns={"consequence": "VEP ANNO"})
    variant_df = variant_df.drop_duplicates()




Processing genes:   0%|          | 10/2806 [00:08<41:12,  1.13it/s]

Rate limit reached. Sleeping 52.1 seconds...


# STEP 5: Calculate if variant position is conserved across Human and C. Elegans.


In [ ]:
from functions.alignment_checker import check_conserved_human_variant_to_worm

variant_df = variant_df[variant_df["VEP ANNO"] == "missense_variant"].copy()
display(variant_df)
variant_df = check_conserved_human_variant_to_worm(variant_df)

variant_df

# STEP 6: Annotate with AlphaMissense predictions

In [ ]:
from functions.alphamissense_mapping import alphamissense_matching


matched_variant_df, alphamissense_subset_df = alphamissense_matching(variant_df)

In [ ]:
out_file = f"created_data/min_{MIN_ALGORITHM_MATCH}_algorithm_match_variant_data.csv"

if not os.path.exists(out_file):
    matched_variant_df.to_csv(out_file, index=False)
else:
    print(f"File already exists, not saving: {out_file}")